# TP2 - Informe tecnico

Sistema de Deteccion y Clasificacion de Razas de Perros — IA 5.2 Computer Vision.

## Equipo
- Gianfranco Frattini
- Matias Prado

## 1. Explicacion completa del pipeline

El sistema implementa un pipeline incremental de Computer Vision para deteccion y
clasificacion de razas de perros, siguiendo el flujo:

**Embeddings -> Busqueda por similitud -> Clasificacion -> Deteccion -> Pipeline completo**

### Etapa 1 — Buscador de imagenes por similitud
1. Se extrae el **embedding** de la imagen de consulta con un modelo pre-entrenado en
   ImageNet (ResNet50 sin capa FC, 2048-d) via `extract_embedding`.
2. Se recuperan los **Top-K vecinos** mas similares de la base vectorial (pgvector o JSON)
   via `search_similar_images`, usando cosine similarity (o L2).
3. Se predice la raza por **voto mayoritario ponderado por score** entre los vecinos
   via `predict_breed_from_neighbors`. Si el mejor score < `SIMILARITY_THRESHOLD` ->
   "unknown".

### Etapa 2 — Clasificacion supervisada
1. Se entrena **ResNet18 fine-tuned** (Modelo A) y una **CNN propia** (Modelo B) sobre
   el dataset 70 Dog Breeds via `train_classifier`.
2. Se evalua con `evaluate_classifier` (accuracy, precision, recall, specificity, F1).
3. `extract_custom_embedding` expone la penultima capa de los modelos entrenados para
   que la busqueda por similitud de la Etapa 1 pueda usarlos (seleccion dinamica).

### Etapa 3 — Deteccion y clasificacion
1. `detect_dogs` usa YOLOv8n pre-entrenado para detectar perros (clase 16 de COCO) y
   devuelve bounding boxes con score de confianza.
2. Cada recorte se clasifica con `classify_detected_dog` usando el modelo entrenado.
3. La orquestacion provista (`DetectionService.predict`) arma el JSON con bbox, raza y
   scores, y el frontend dibuja los resultados sobre la imagen.

### Integracion
- Backend: FastAPI asincronico (`/search`, `/classify`, `/detect`, `/status`, `/models`).
- Frontend: Gradio con una pestana por etapa.
- Base vectorial: PostgreSQL + pgvector (default) o JSON file (`USE_PGVECTOR=false`).
- Configuracion: todo via `.env` (modelo, threshold, top-k, paths, YOLO).
- Despliegue: Docker Compose (backend + postgres + frontend).

## 2. Dataset

**Dataset principal:** [70 Dog Breeds Image Dataset (Kaggle)](https://www.kaggle.com/datasets/gpiosenka/70-dog-breedsimage-data-set)

- **70 razas** de perros.
- Splits provistos: `train/` (7946 imagenes), `valid/` (700 imagenes), `test/` (700 imagenes),
  cada uno con una subcarpeta por raza.
- Distribucion: ver el EDA detallado en `etapa2_colab.ipynb` seccion 5.
- **Conjunto independiente para evaluacion**: imagenes descargadas de internet
  (fuera del dataset de Kaggle) en `data/external/<raza>/`. Se usa para evaluar la
  generalizacion del sistema fuera de la distribucion del dataset.
- **Filtrado de baja calidad**: se descartan imagenes corruptas/no legibles durante
  el EDA (cv2/PIL no pueden abrirlas).

## 3. Preprocesamiento

| Tecnica | Valor | Justificacion |
|---|---|---|
| Resize | 224x224 | Tamano estandar ResNet/ImageNet; las imagenes tienen tamanos variables. |
| Normalizacion | mean/std ImageNet | El modelo pre-entrenado espera esa distribucion. |
| Horizontal Flip | p=0.5 | El perro no cambia de raza al espejar; duplica la data. |
| Rotacion | +/-15 grados | Tolerancia a inclinaciones de camara. |
| GaussianBlur | p=0.3, k=3 | Robustez a desenfoque/baja calidad. |
| Brillo/Contraste | jitter 0.2 | Compensa iluminacion variable. |
| RandomResizedCrop | scale 0.8-1.0 | Simula distintos encuadres y distancias. |
| Filtrado baja calidad | manual | Se descartan imagenes no legibles en el EDA. |

El data augmentation **solo** se aplica al split de train; valid/test usan solo
resize + normalizacion (para una evaluacion justa).

## 4. Justificacion de los modelos elegidos

### Modelo de embeddings baseline (Etapa 1): ResNet50
- Pre-entrenado en ImageNet, sin capa FC -> embedding de 2048-d.
- Elegido por sus features ricas y generalizables; mayor capacidad que ResNet18 para
  distinguish razas via similitud coseno.
- Trade-off: mayor consumo de memoria/inferencia que ResNet18, pero aceptable para
  un sistema offline (no tiempo real critico).

### Modelo A (Etapa 2): ResNet18 fine-tuned
- Transfer learning: parte de pesos pre-entrenados en ImageNet -> convergencia rapida
  y alta accuracy con pocos datos por raza.
- ~11M parametros: buen balance precision/costo. Ideal para GPU de consumo (RTX 3080)
  o Colab T4.
- Embedding de 512-d (antes de FC) reutilizable en la Etapa 1 via
  `extract_custom_embedding`.

### Modelo B (Etapa 2): CNN propia
- Bloques conv + BatchNorm + ReLU + MaxPool + GAP + FC (512-d, ~1-2M parametros).
- Entrenada desde cero: sirve como baseline de comparacion para medir el valor del
  transfer learning. Mas liviana pero necesita mas datos/epochs para converger.

### Detector (Etapa 3): YOLOv8n
- Pre-entrenado en COCO, clase 'dog' = 16. No requiere entrenamiento.
- YOLOv8n es la variante mas liviana (rapida inferencia) y suficiente para detectar
  perros en escenas complejas (uno o multiples).
- Trade-off: podria usar YOLOv8s/m para mas precision, pero n basta para el TP.

### Base vectorial: pgvector
- PostgreSQL + pgvector permite busqueda coseno eficiente (operador `<=>`) y escala
  mejor que un JSON file. Alternativa JSON (`USE_PGVECTOR=false`) para entornos sin
  docker (ej. Colab).

## 5. Proceso de entrenamiento e hiperparametros

Implementado en `ClassifierService.train_classifier` (orquestado desde
`etapa2_colab.ipynb` o `scripts/train_classifier.py`).

**Hiperparametros (Modelo A y B):**
- Optimizador: AdamW (lr=1e-3, weight_decay=1e-4)
- Scheduler: ReduceLROnPlateau (mode=min, patience=2, factor=0.5)
- Batch size: 32
- Epochs: 25 (con early stopping patience=5 sobre val_loss)
- Loss: CrossEntropy
- Data augmentation: seccion 3 (solo train)
- Device: CUDA (RTX 3080)

**Artefactos generados en `training_results/<modelo>/`:**
- `history.json` — curvas train/val loss y accuracy por epoch.
- `metrics.json` — accuracy, precision, recall, specificity, F1 sobre test.
- `training_curves.png` — loss y accuracy (train vs val) por epoch.
- `confusion_matrix.png` — matriz de confusion sobre test.

Las curvas y matrices de confusion se muestran a continuacion.

## 6. Resultados obtenidos

### Etapa 1 — NDCG@10

Se evaluo la busqueda por similitud sobre las 700 imagenes del split `test/` con el
modelo baseline (ResNet50, 2048-d). Para cada imagen se recuperaron Top-10 vecinos
y se calculo NDCG@10 con relevancia binaria (1.0 si el vecino es de la misma raza
que la consulta, 0.0 si no).

**NDCG@10 medio = 0.9674**

Este valor alto (cercano a 1.0) indica que la busqueda por similitud con embeddings
de ResNet50 recupera correctamente vecinos de la misma raza en casi todos los casos.
El modelo pre-entrenado en ImageNet produce features lo suficientemente discriminativas
para distinguir razas de perros sin necesidad de fine-tuning, lo cual valida el enfoque
de la Etapa 1.

Script: `scripts/evaluate_search.py --split test`

In [ ]:
from IPython.display import Image, display
import json

# Cargar metricas reales
with open('training_results/resnet18_finetuned/metrics.json') as f:
    metrics_a = json.load(f)
with open('training_results/cnn_custom/metrics.json') as f:
    metrics_b = json.load(f)

print('=== Etapa 2 — Tabla de metricas ===')
print(f"{'Modelo':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'Specificity':>12} {'F1':>10}")
print('-' * 77)
print(f"{'ResNet18 fine-tuned':<25} {metrics_a['accuracy']:>10.4f} {metrics_a['precision']:>10.4f} {metrics_a['recall']:>10.4f} {metrics_a['specificity']:>12.4f} {metrics_a['f1']:>10.4f}")
print(f"{'CNN custom':<25} {metrics_b['accuracy']:>10.4f} {metrics_b['precision']:>10.4f} {metrics_b['recall']:>10.4f} {metrics_b['specificity']:>12.4f} {metrics_b['f1']:>10.4f}")

### Etapa 2 — Curvas de entrenamiento

**Modelo A: ResNet18 fine-tuned** — 25 epochs completos. El train_loss baja de 1.97 a
0.02 mientras el val_loss se estabiliza alrededor de 0.70 (mejor en epoch 22: 0.703).
El train_acc llega a 99.4% y el val_acc a 92.6%, mostrando un ligero overfitting
hacia el final que el early stopping (patience=5) controla correctamente.

**Modelo B: CNN custom** — 25 epochs completos. Convergencia mucho mas lenta: el
train_loss baja de 3.86 a 1.10 y el val_acc llega a 67.6%. Sin transfer learning,
la red necesita mas epochs para alcanzar un desempeno comparable.

In [ ]:
# Curvas de entrenamiento - ResNet18 fine-tuned
print('Modelo A: ResNet18 fine-tuned — Curvas de entrenamiento')
display(Image(filename='training_results/resnet18_finetuned/training_curves.png'))

In [ ]:
# Curvas de entrenamiento - CNN custom
print('Modelo B: CNN custom — Curvas de entrenamiento')
display(Image(filename='training_results/cnn_custom/training_curves.png'))

### Etapa 2 — Matrices de confusion

**ResNet18 fine-tuned** logra 94.1% de accuracy en test. Los errores se concentran
en razas visualmente similares (ej. variantes de terrier, spaniel). La matriz
muestra una diagonal muy marcada con pocos falsos positivos fuera de ella.

**CNN custom** logra 65.6% de accuracy. La matriz muestra mas dispersion,
especialmente entre razas morfologicamente similares, reflejando la menor capacidad
de la red sin transfer learning.

In [ ]:
# Matriz de confusion - ResNet18 fine-tuned
print('Modelo A: ResNet18 fine-tuned — Matriz de confusion (test)')
display(Image(filename='training_results/resnet18_finetuned/confusion_matrix.png'))

In [ ]:
# Matriz de confusion - CNN custom
print('Modelo B: CNN custom — Matriz de confusion (test)')
display(Image(filename='training_results/cnn_custom/confusion_matrix.png'))

### Etapa 1 — Visualizacion de embeddings (PCA y t-SNE)

Para analizar la calidad de los embeddings del modelo baseline (ResNet50, 2048-d),
se extrajeron embeddings de 350 imagenes del split test (5 por raza) y se redujeron
a 2D con **PCA** y **t-SNE**.

- **PCA**: los dos primeros componentes explican ~10% de la varianza (PC1=5.7%,
  PC2=4.3%), lo cual es esperable en embeddings de alta dimension. Aun asi, se
  observan ciertos clusters por raza.
- **t-SNE**: preserva la estructura local y revela clusters mas definidos por raza.
  Razas morfologicamente similares (ej. terriers, spaniels) tienden a agruparse
  cerca, mientras que razas muy distintas (ej. Afghan, Husky) forman clusters
  bien separados.

Scripts: `scripts/visualize_embeddings.py` (genera los PNGs en
`training_results/embeddings/`). Notebook detallada: `etapa1_colab.ipynb`.

In [ ]:
# PCA de embeddings baseline
print('PCA de embeddings baseline (ResNet50, 2048-d) — 70 razas')
display(Image(filename='training_results/embeddings/pca_embeddings.png'))

In [ ]:
# t-SNE de embeddings baseline
print('t-SNE de embeddings baseline (ResNet50, 2048-d) — 70 razas')
display(Image(filename='training_results/embeddings/tsne_embeddings.png'))

## 7. Comparacion entre enfoques

### Busqueda por similitud vs clasificacion supervisada
- **Busqueda por similitud** (Etapa 1) no requiere entrenamiento: usa features de un
  modelo pre-entrenado y recupera vecinos. NDCG@10 = 0.9674, excelente sin entrenar.
  La raza predicha por voto mayoritario es correcta en la mayoria de los casos,
  pero depende de que existan ejemplos similares en la base.
- **Clasificacion supervisada** (Etapa 2) aprende un mapeo directo imagen->raza y
  logra 94.1% de accuracy con ResNet18 fine-tuned, con confianza (softmax) mas
  interpretable. A costa de requerir entrenamiento y datos etiquetados.
- La integracion permite usar los modelos fine-tuned como extractores de embeddings
  en la Etapa 1 (seleccion dinamica), combinando lo mejor de ambos enfoques.

### ResNet18 fine-tuned vs CNN custom

| Metrica | ResNet18 fine-tuned | CNN custom | Diferencia |
|---|---|---|---|
| Accuracy | 0.9414 | 0.6557 | +0.2857 |
| Precision | 0.9459 | 0.7252 | +0.2207 |
| Recall | 0.9414 | 0.6557 | +0.2857 |
| Specificity | 0.9992 | 0.9950 | +0.0042 |
| F1 | 0.9414 | 0.6458 | +0.2956 |

- ResNet18 fine-tuned supera a la CNN custom por ~28.6 puntos de accuracy gracias al
  transfer learning de ImageNet. Las features pre-entrenadas (bordes, texturas,
  patrones) son altamente reutilizables para distinguir razas.
- La CNN custom, entrenada desde cero con ~11k imagenes, no logra converger en 25
  epochs (val_acc aun subia al final). Necesitaria mas epochs o mas datos.
- **Specificity** es alta en ambos (99.5%+) porque al ser clasificacion multiclase
  con 70 clases, cada muestra tiene muchas clases verdaderas negativas.
- La CNN custom es mas liviana (~1-2M parametros vs ~11M) pero el costo computacional
  no justifica la perdida de precision en este caso.
- Ver los graficos de curvas y matrices de confusion arriba para el detalle visual.

## 8. Problemas encontrados y soluciones implementadas

- **`extract_custom_embedding` removido del enunciado**: la catedra (INFORMACION-EXTRA.md)
  indico ignorar esta funcion, pero la infraestructura provista (`api.py`,
  `build_index.py`) la invoca al seleccionar un modelo fine-tuned en la Etapa 1, y el
  enunciado pide seleccion dinamica de modelos de embeddings. Se decidio **implementarla**
  para que el sistema funcione end-to-end, documentando la decision (ver seccion 9).

- **PyTorch CPU-only**: el `requirements.txt` instala la version CPU de torch por
  defecto. Para entrenar con GPU (RTX 3080) hubo que reinstalar torch desde el index
  URL de CUDA (`--index-url https://download.pytorch.org/whl/cu128`).

- **Paths Windows en pgvector**: al indexar desde Windows, los paths se guardaban con
  backslashes (`..\data\dataset\...`) y el contenedor Docker (Linux) no podia servir
  las imagenes similares. Solucion: script `reindex_docker.py` que indexa desde el
  host con GPU guardando paths Linux (`data/dataset/train/...`).

- **Error de tipo en pgvector**: `operator does not exist: vector <=> double precision[]`.
  Al pasar una `list[float]` a `store.search`, psycopg la enviaba como array nativo
  de Postgres y el operador `<=>` de pgvector no resolvia el cast. Solucion: convertir
  el query a `np.ndarray` antes de pasarlo al store (el dumper de `register_vector`
  lo convierte a tipo `vector`).

- **Docker: contenedores viejos con mismo nombre**: contenedores de un TP anterior
  ocupaban los nombres `backend` y `frontend`. Solucion: `docker rm -f` antes de
  `docker compose up`.

- **Multiprocessing en Windows (graphify)**: el pool de procesos requiere
  `if __name__ == '__main__'` + `freeze_support()` para no romper en spawn.

- **Checkpoint con metadatos**: el `load_model` provisto usa `weights_only=False`, lo
  que permite guardar un dict con `classes`, `arch`, `history` ademas del `state_dict`,
  facilitando la reconstruccion del modelo en inferencia.

## 9. Modificaciones fuera de las funciones indicadas

### Justificadas

1. **`extract_custom_embedding` implementada** (Etapa 2): la catedra indico ignorarla,
   pero `src/lib/api.py:61` (`_embedding_extractor`) y `scripts/build_index.py:50` la
   invocan al seleccionar un modelo `resnet18_finetuned` o `cnn_custom` en la Etapa 1,
   y el enunciado pide seleccion dinamica de modelos de embeddings. Sin implementarla,
   la pestana Etapa 1 rompe al elegir un modelo fine-tuned. Se documenta la decision.

2. **Archivos nuevos (aditivos, no tocan infraestructura provista):**
   - `src/.env` — copia local de `.env.local.example` (ya en `.gitignore`).
   - `scripts/evaluate_search.py` — NDCG@10 de la Etapa 1 sobre un conjunto independiente.
   - `scripts/reindex_docker.py` — reindexa el dataset guardando paths Linux relativos
     (`data/dataset/train/...`) para que el frontend docker sirva las imagenes similares.
     Se ejecuta **una sola vez** desde el host con GPU; los embeddings persisten en el
     volumen docker `pgdata` y sobreviven a cualquier `docker compose down/up`.
   - `scripts/visualize_embeddings.py` — genera visualizaciones PCA y t-SNE de los
     embeddings del modelo baseline en `training_results/embeddings/`.
   - `tests/` — tests basicos pytest (dimensiones de embedding, voto mayoritario,
     `_clip_xyxy`, formato de checkpoint).
   - `training_results/` — carpeta donde `train_classifier` vuelca `history.json`,
     `metrics.json`, `training_curves.png` y `confusion_matrix.png` por modelo.
   - `etapa1_colab.ipynb` — notebook de la Etapa 1 (PCA/t-SNE, NDCG@10, analisis).
   - `etapa3_colab.ipynb` — notebook de la Etapa 3 (deteccion YOLO, visualizacion).
   - `EJECUCION.md` — guia de ejecucion paso a paso.

3. **Helpers aditivos dentro de los servicios** (no reemplazan funciones provistas):
   - `similarity_service.py`: cache lazy del ResNet50 + `_cv2_to_pil`; casteo del query
     a `np.ndarray` en `search_similar_images` para que pgvector reciba tipo `vector`.
   - `classifier_service.py`: `_build_resnet18`, `_load_classifier_model`,
     `_ResNet18Wrapper`, `_CNNCustom`, `_build_cnn_custom`, `_bgr_to_rgb_pil`,
     `_save_training_results`.
   - `detection_service.py`: cache lazy del modelo YOLO.

4. **`.env.local.example` y `.env.docker.example`**: `EMBEDDING_DIM` 512 -> 2048 para
   coincidir con el modelo baseline (ResNet50). Es un cambio de configuracion, no de
   codigo, y esta permitido por el enunciado (se configuran los modelos via `.env`).

### Flujo de reindexado al cambiar de modelo de embeddings

La base vectorial se indexa con un unico modelo a la vez (la columna `vector(EMBEDDING_DIM)`
de pgvector tiene una dimension fija). Por defecto se usa `baseline` (ResNet50, 2048-d).
Para que la Etapa 1 funcione con `resnet18_finetuned` o `cnn_custom` (512-d) hay que:
1. En `.env.docker.example` (y `src/.env` local) cambiar `EMBEDDING_MODEL=resnet18_finetuned`
   y `EMBEDDING_DIM=512`.
2. Reindexar: `python scripts/reindex_docker.py` (desde el host con GPU, una sola vez;
   los nuevos embeddings persisten en el volumen `pgdata`).
3. `docker compose up` (el backend lee la nueva config y la base ya esta indexada).

Mientras se mantenga el mismo modelo, **no hace falta reindexar** al reiniciar docker:
el volumen `pgdata` conserva los embeddings entre `docker compose down` y `up`.

Ninguna funcion provista fue modificada; solo se completaron las 8 funciones indicadas
y se agregaron helpers/modulos auxiliares para soportarlas.